<a href="https://colab.research.google.com/github/Karolina-000/data-science-practice/blob/main/SQL_Subqueries_Films.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect(":memory:")

con.executescript("""
CREATE TABLE films (
    id     INTEGER PRIMARY KEY,
    title  TEXT,
    genre  TEXT,
    year   INTEGER,
    rating REAL
);
CREATE TABLE reviews (
    id      INTEGER PRIMARY KEY,
    film_id INTEGER,
    score   INTEGER
);
""")

con.executemany("INSERT INTO films VALUES (?,?,?,?,?)", [
    (1, "ცისფერი მთები",    "კომედია", 1983, 8.6),
    (2, "მონანიება",        "დრამა",   1984, 8.2),
    (3, "ოთარაანთ ქვრივი",  "დრამა",   1957, 7.4),
    (4, "ჯარისკაცის მამა",  "დრამა",   1964, 7.9),
    (5, "კეთილი იყოს",      "კომედია", 1998, 6.8),
    (6, "ხევსურული ბალადა", "დრამა",   1965, 7.1),
    (7, "ახალი ფილმი",      "კომედია", 2024, 5.9),
])

con.executemany("INSERT INTO reviews VALUES (?,?,?)", [
    (1, 1, 9), (2, 1, 8), (3, 2, 9), (4, 3, 7), (5,  1, 10),
    (6, 2, 8), (7, 4, 8), (8, 5, 6), (9, 4, 7), (10, 2, 9),
])

con.commit()
print("ბაზა მზადაა: 7 ფილმი · 10 რეცენზია")

ბაზა მზადაა: 7 ფილმი · 10 რეცენზია


In [ ]:
from IPython.display import display, HTML

def show(title, sql):
    df = pd.read_sql_query(sql, con)
    html = df.to_html(index=False, float_format=lambda x: f"{x:g}", border=0)

    display(HTML(f"""
    <div style="margin:0; padding:14px 18px; background:#0d0906;">
      <style>
        .ttl {{
          display:inline-block; background:#3b2a5c; color:#e6dcff;
          font-size:15px; letter-spacing:1px; padding:6px 22px 6px 14px;
          border-bottom:2px solid #8a4fff;
          clip-path:polygon(0 0, 100% 0, 92% 100%, 0 100%);
        }}
        .line {{ height:1px; background:#5b3fa0; margin:0 0 10px 0; }}
        .tbl table {{ border-collapse:collapse; min-width:340px; font-size:15px; }}
        .tbl th {{
          background:#2a1f45; color:#b78cff; padding:8px 18px; text-align:left;
          border-bottom:2px solid #8a4fff;
        }}
        .tbl td {{
          padding:7px 18px; color:#e6e1ee; background:#0d0906;
          border-bottom:1px solid #2e2540;
        }}
        .tbl tr:nth-child(even) td {{ background:#161019; }}
        .tbl tr:hover td {{ background:#3b2a5c; color:#ffffff; }}
      </style>
      <div class="ttl">{title}</div>
      <div class="line"></div>
      <div class="tbl">{html}</div>
    </div>
    """))

show("დავალება 1 ა) საშუალო რეიტინგი", """
SELECT ROUND(AVG(rating), 3) AS საშუალო_რეიტინგი
FROM films
""")

show("დავალება 1 ბ) ფილმები საშუალოზე მაღალი რეიტინგით", """
SELECT title AS სათაური, rating AS რეიტინგი
FROM films
WHERE rating > (SELECT AVG(rating) FROM films)
ORDER BY rating DESC
""")

show("დავალება 2 ა) ყველაზე მაღალრეიტინგული ფილმი", """
SELECT title AS სათაური, genre AS ჟანრი, rating AS რეიტინგი
FROM films
WHERE rating = (SELECT MAX(rating) FROM films)
""")

show("დავალება 2 ბ) ყველაზე ძველი ფილმი", """
SELECT title AS სათაური, year AS წელი
FROM films
WHERE year = (SELECT MIN(year) FROM films)
""")

show("დავალება 3 ა) ფილმები რეცენზიით", """
SELECT title AS სათაური
FROM films
WHERE id IN (SELECT film_id FROM reviews)
""")

show("დავალება 3 ბ) ფილმები რეცენზიის გარეშე", """
SELECT title AS სათაური, year AS წელი
FROM films
WHERE id NOT IN (SELECT film_id FROM reviews)
""")

show("დავალება 4) ფილმები 9 ან 10 ქულით", """
SELECT title AS სათაური
FROM films
WHERE id IN (
    SELECT film_id
    FROM reviews
    WHERE score >= 9
)
""")

show("დავალება 5) სხვაობა საშუალოსგან", """
SELECT
    title AS სათაური,
    rating AS რეიტინგი,
    ROUND(rating - (SELECT AVG(rating) FROM films), 2) AS სხვაობა
FROM films
ORDER BY rating DESC
""")

საშუალო_რეიტინგი
7.414


სათაური,რეიტინგი
ცისფერი მთები,8.6
მონანიება,8.2
ჯარისკაცის მამა,7.9


სათაური,ჟანრი,რეიტინგი
ცისფერი მთები,კომედია,8.6


სათაური,წელი
ოთარაანთ ქვრივი,1957


სათაური
ცისფერი მთები
მონანიება
ოთარაანთ ქვრივი
ჯარისკაცის მამა
კეთილი იყოს


სათაური,წელი
ხევსურული ბალადა,1965
ახალი ფილმი,2024


სათაური
ცისფერი მთები
მონანიება


სათაური,რეიტინგი,სხვაობა
ცისფერი მთები,8.6,1.19
მონანიება,8.2,0.79
ჯარისკაცის მამა,7.9,0.49
ოთარაანთ ქვრივი,7.4,-0.01
ხევსურული ბალადა,7.1,-0.31
კეთილი იყოს,6.8,-0.61
ახალი ფილმი,5.9,-1.51
